In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error
import xgboost as xgb
import shap
import matplotlib
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

y = df['Churn']
X = df.drop(columns=['Churn'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
churn_model = RandomForestClassifier(n_estimators=100, random_state=42)
churn_model.fit(X_train, y_train)
df['churn_probability'] = churn_model.predict_proba(X)[:, 1]
df['churn_risk'] = df['churn_probability'].apply(lambda x: 'HIGH' if x > 0.7 else ('MEDIUM' if x > 0.4 else 'LOW'))

df['lead_score'] = (
    (df['tenure'] / df['tenure'].max()) * 40 +
    (df['MonthlyCharges'] / df['MonthlyCharges'].max()) * 30 +
    df['Contract'].map({0: 0, 1: 15, 2: 30})
)

X_lead = df.drop(columns=['Churn', 'lead_score', 'churn_probability', 'churn_risk'])
y_lead = df['lead_score']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_lead, y_lead, test_size=0.2, random_state=42)
lead_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
lead_model.fit(X_train_l, y_train_l)
df['lead_score_predicted'] = lead_model.predict(X_lead)

explainer = shap.Explainer(lead_model)
shap_values = explainer(X_lead)

model_st = SentenceTransformer('all-MiniLM-L6-v2')

def customer_to_text(row):
    churn_status = "churned" if row['Churn'] == 1 else "stayed as customer"
    return (
        f"Customer with {row['tenure']} months tenure, "
        f"${row['MonthlyCharges']:.0f} monthly charges, "
        f"lead score {row['lead_score']:.0f}/100, "
        f"and {churn_status}."
    )

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name="customers")
except:
    pass
collection = chroma_client.create_collection(name="customers")
texts = [customer_to_text(row) for _, row in df.iterrows()]
batch_size = 5000
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    batch_embeddings = model_st.encode(batch_texts).tolist()
    batch_ids = [str(j) for j in range(i, i+len(batch_texts))]
    collection.add(documents=batch_texts, embeddings=batch_embeddings, ids=batch_ids)

def find_similar_customers(index, top_n=3):
    query_text = customer_to_text(df.iloc[index])
    query_embedding = model_st.encode([query_text]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_n+1)
    similar = []
    for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
        if results['ids'][0][i] == str(index):
            continue
        similarity = round((1 - distance) * 100, 1)
        similar.append({"text": doc, "similarity": similarity})
    return similar[:top_n]

import os
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY", "YOUR_GROQ_KEY_HERE"))

def generate_email(index):
    row = df.iloc[index]
    similar = find_similar_customers(index)
    similar_customers_text = "\n".join([f"- {s['text']}" for s in similar])
    prompt = f"""
You are a helpful sales assistant. Write a short, warm, personalised email to a customer with these details:
- Tenure: {row['tenure']} months
- Monthly charges: ${row['MonthlyCharges']}
- Churn risk: {row['churn_probability']*100:.0f}%
- Lead score: {row['lead_score']:.0f}/100

Here are 3 similar past customers:
{similar_customers_text}

If churn risk is high (>70%), focus on retention.
If lead score is high (>75%), focus on upselling.
Keep it under 100 words. Sound human, not robotic.
"""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print("All models loaded successfully!")
print("Total customers:", len(df))
print("Churn risk distribution:\n", df['churn_risk'].value_counts())

c:\Users\HARSHADA\Desktop\sales-ai-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1656.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded successfully!
Total customers: 7032
Churn risk distribution:
 churn_risk
LOW       5138
HIGH      1368
MEDIUM     526
Name: count, dtype: int64
